# SKEX on Kaggle

Settings: **Internet on**, **one T4**. Do not enable a second GPU.

Add Data -> your dataset named **`skex-datasets`**. Kaggle mounts a zip at `/kaggle/input/<slug>/`. The slug must be exactly `skex-datasets`, so the files land here:

- `/kaggle/input/skex-datasets/processed/domain_a/{train,dev,test}.jsonl` — what the runner loads
- `/kaggle/input/skex-datasets/sciriff/4096/` — SciRIFF parquet
- `/kaggle/input/skex-datasets/scier/` — SciER JSONL
- `/kaggle/input/skex-datasets/scierc/extracted/processed_data/json/` — SciERC JSONL
- `/kaggle/input/skex-datasets/cord/text/` — CORD receipt text

This notebook clones `https://github.com/umardrazbhatti-work/skex`, checks those paths, and copies Domain A into the clone. It does not download a 7B model, install Unsloth, or start plan 01.


In [ ]:
import shutil
import subprocess
from pathlib import Path

if shutil.which("nvidia-smi"):
    listing = subprocess.check_output(["nvidia-smi", "-L"], text=True)
    print(listing)
    gpus = [line for line in listing.splitlines() if line.strip()]
    if len(gpus) != 1:
        raise SystemExit(f"Stop. Expected one T4, found {len(gpus)}. Turn the extra GPU off before any training run.")
    if "T4" not in listing:
        print("WARNING: the visible GPU is not named T4. Do not start a larger run on this session.")
else:
    print("No NVIDIA GPU visible. The smoke cells below do not need one.")


In [ ]:
import subprocess
import sys
from pathlib import Path

REMOTE = "https://github.com/umardrazbhatti-work/skex.git"

def run(cmd, cwd):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd))

if Path("/kaggle/working").is_dir():
    work = Path("/kaggle/working")
    repo = work / "skex"
    if not (repo / ".git").exists():
        run(["git", "clone", REMOTE], work)
    else:
        run(["git", "pull", "--ff-only"], repo)
elif (Path.cwd() / "src" / "skex").is_dir():
    repo = Path.cwd()
    print(f"Already inside {repo}. Not cloning.")
else:
    raise SystemExit("Run this notebook on Kaggle, or from the skex repo root.")

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], repo)
print("repo", repo)


In [ ]:
import json
import os
import shutil
from pathlib import Path

DATASET_SLUG = "skex-datasets"
DATA = Path("/kaggle/input") / DATASET_SLUG

PATHS = {
    "domain_a": DATA / "processed" / "domain_a",
    "sciriff": DATA / "sciriff" / "4096",
    "scier_llm": DATA / "scier" / "LLM",
    "scier_plm": DATA / "scier" / "PLM",
    "scierc": DATA / "scierc" / "extracted" / "processed_data" / "json",
    "cord": DATA / "cord" / "text",
}
REQUIRED = [
    PATHS["domain_a"] / "train.jsonl",
    PATHS["domain_a"] / "dev.jsonl",
    PATHS["domain_a"] / "test.jsonl",
    PATHS["sciriff"] / "train-00000-of-00001.parquet",
    PATHS["sciriff"] / "validation-00000-of-00001.parquet",
    PATHS["sciriff"] / "test-00000-of-00001.parquet",
    PATHS["scier_llm"] / "train.jsonl",
    PATHS["scier_llm"] / "dev.jsonl",
    PATHS["scier_llm"] / "test.jsonl",
    PATHS["scier_plm"] / "train.jsonl",
    PATHS["scierc"] / "train.json",
    PATHS["scierc"] / "dev.json",
    PATHS["scierc"] / "test.json",
    PATHS["cord"] / "train.jsonl",
    PATHS["cord"] / "dev.jsonl",
    PATHS["cord"] / "test.jsonl",
]

if not DATA.is_dir():
    attached = sorted(p.name for p in Path("/kaggle/input").iterdir()) if Path("/kaggle/input").is_dir() else []
    raise SystemExit(
        "Dataset slug not found at /kaggle/input/skex-datasets.\n"
        "Create the Kaggle dataset with the name skex-datasets (that exact slug), "
        "upload skex-datasets.zip, and Add Data on this notebook.\n"
        f"Attached inputs: {attached or 'none'}"
    )

missing = [str(path) for path in REQUIRED if not path.is_file()]
if missing:
    raise SystemExit("Attached dataset is missing files:\n" + "\n".join(missing))

repo = Path("/kaggle/working/skex") if Path("/kaggle/working/skex").is_dir() else Path.cwd()
dest = repo / "data" / "processed" / "domain_a"
dest.mkdir(parents=True, exist_ok=True)
for name in ("train.jsonl", "dev.jsonl", "test.jsonl"):
    shutil.copy2(PATHS["domain_a"] / name, dest / name)
    n = sum(1 for line in (dest / name).open(encoding="utf-8") if line.strip())
    print(f"domain_a {name}: {n} rows <- {PATHS['domain_a'] / name}")

for label, folder in PATHS.items():
    print(f"{label}: {folder} exists={folder.is_dir()}")

registered = {key: str(path) for key, path in PATHS.items()}
registered["domain_a_repo"] = str(dest)
(repo / "data_paths.json").write_text(json.dumps(registered, indent=2), encoding="utf-8")
os.environ["SKEX_DATA_ROOT"] = str(DATA)
print("registered", json.dumps(registered, indent=2))
print("CORD text is registered but not copied into data/processed/domain_b. That converter is not written yet.")


In [ ]:
import subprocess
import sys
from pathlib import Path

repo = Path("/kaggle/working/skex") if Path("/kaggle/working/skex").is_dir() else Path.cwd()
subprocess.check_call([sys.executable, "-m", "pytest", "-q"], cwd=str(repo))
for _ in range(2):
    subprocess.check_call(
        [sys.executable, "-m", "skex.experiments.runner", "--plan", "experiments/plans/00_smoke.yaml"],
        cwd=str(repo),
    )
subprocess.check_call([sys.executable, "-m", "skex.experiments.status"], cwd=str(repo))
print("Stop here. Do not run experiments/plans/01_tax_zeroshot.yaml or install Unsloth in this notebook until you choose to spend T4 quota.")
